In [1]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [2]:
metrics_dir=os.path.join(os.getcwd(),"metrics")
# --- LOAD THE .pkl FILE GENERATED PREVIOUSLY ---
adapt=0
test_ds_b4=False
if test_ds_b4:
    identifier=f"testing_dsb4_adaptable{adapt}" if adapt > 0 else "testing_dsb4"
else:
    identifier=f"30000_1000_100_adaptable{adapt}" if adapt > 0 else "30000_1000_100"
    # identifier="1000_200_median"


In [ ]:
filename= f"channel_metrics_by_network_{identifier}_3.csv"
results_df_3= pd.read_csv(os.path.join(metrics_dir, filename))
print(f"Results loaded from {os.path.join(metrics_dir, filename)}")
print(results_df_3.head())

filename= f"channel_metrics_by_network_{identifier}.csv"
results_df=pd.read_csv(os.path.join(metrics_dir, filename))
print(f"Results loaded from {os.path.join(metrics_dir, filename)}")
print(results_df.head())

Results loaded from c:\Users\Pc\Documents\Tese\LAVA_SNN_ripples\snnTorch\eval\live_results\metrics\channel_metrics_by_network_30000_1000_100_3.csv
                   Network                     Dataset   TP   FP  FN  \
0  dsb4updn_median_200_15f  Amigo2_2019-07-11_11-57-07  194  321  30   
1  dsb4updn_median_200_15f    Dlx1_2021-02-12_12-46-54   27   49   4   
2  dsb4updn_median_200_15f    Som2_2019-07-24_12-01-49   54  102  10   
3  dsb4updn_median_200_15f    Thy7_2020-11-11_16-05-00  111   24  28   
4  dsb4updn_median_200_11b  Amigo2_2019-07-11_11-57-07  192  303  32   

   Precision    Recall        F1  
0   0.376699  0.866071  0.525034  
1   0.355263  0.870968  0.504673  
2   0.346154  0.843750  0.490909  
3   0.822222  0.798561  0.810219  
4   0.387879  0.857143  0.534075  
Results loaded from c:\Users\Pc\Documents\Tese\LAVA_SNN_ripples\snnTorch\eval\live_results\metrics\channel_metrics_by_network_30000_1000_100.csv
                   Network                     Dataset  Channel  

In [5]:
df_chan_avg = results_df.groupby(['Network', 'Dataset'])[["Precision", "Recall", "F1"]].mean().reset_index()
df_compare = pd.merge(results_df_3, df_chan_avg, on=['Network', 'Dataset'], suffixes=('_agg', '_chan'))

In [6]:
from scipy.stats import ttest_rel, wilcoxon

results = []
metrics = ["Precision", "Recall", "F1"]

for metric in metrics:
    agg_vals = df_compare[f"{metric}_agg"]
    chan_vals = df_compare[f"{metric}_chan"]

    # Paired t-test
    stat, pval = ttest_rel(agg_vals, chan_vals)
    results.append({
        "Metric": metric,
        "t_stat": stat,
        "p_value": pval,
        "Agg_mean": agg_vals.mean(),
        "Chan_mean": chan_vals.mean()
    })

results_df = pd.DataFrame(results)
print(results_df)


      Metric     t_stat       p_value  Agg_mean  Chan_mean
0  Precision   2.362656  2.233529e-02  0.561845   0.551778
1     Recall  18.559149  2.862665e-23  0.790460   0.646072
2         F1   9.243076  3.813207e-12  0.626113   0.541775
